# AnimalCLEF 2026: Ensemble Re-ID V5.1 (Fine-tuned MegaDescriptor)

**Approach**: Combines dual global features (MiewID v3 + MegaDescriptor-L-384) with species-specific local features (SIFT, KAZE, ALIKED) using weighted ensemble voting.

**Expected Improvements**:
- SeaTurtleID2022: 38% → 50-60% known match rate
- Better clustering for deformable bodies (Salamanders) and dense patterns (Texas Lizards)

**Runtime**: ~90 minutes (within Kaggle's 9-hour limit)

## Section 1: Setup and Installation

In [ ]:
# Cell 1.1: Install Dependencies
!pip install kornia kornia-rs kornia-moons --quiet
# LightGlue is NOT on PyPI — install directly from git
!pip install --quiet git+https://github.com/cvg/LightGlue.git

# Verify by importing the actual classes (lightglue has no __version__)
from lightglue import ALIKED as _aliked_test, LightGlue as _lg_test
del _aliked_test, _lg_test
print("\u2713 LightGlue installed successfully")

!pip install wildlife-datasets wildlife-tools timm scikit-learn --quiet
!pip install transformers==4.36.0 --quiet
# Force-reinstall huggingface_hub 0.36.2 (last 0.x, has reset_sessions).
# --force-reinstall + --upgrade replaces the system copy, not user site.
!pip install "huggingface_hub==0.36.2" --force-reinstall --upgrade --quiet

import kornia
print(f"\u2713 Kornia {kornia.__version__} installed")

# Flush any stale huggingface_hub cached by Kaggle before notebook start
import sys as _sys
_hf_stale = [k for k in list(_sys.modules) if k.startswith('huggingface_hub')]
for _k in _hf_stale:
    del _sys.modules[_k]
del _sys, _hf_stale

import transformers
print(f"\u2713 Transformers {transformers.__version__} installed")
print("\u2713 All dependencies installed successfully")

In [ ]:
# Cell 1.2: Imports
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import torchvision.transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
import cv2
from PIL import Image

# Deep learning
import timm
from transformers import AutoModel

# Local features (kornia for all extractors)
import kornia.feature as KF

# Dataset
from wildlife_datasets.datasets import AnimalCLEF2026

warnings.filterwarnings("ignore")
print("✓ Imports successful")

In [ ]:
# Cell 1.3: Configuration - Species-Specific Weights

# V5.1: Fine-tuned MegaDescriptor (ArcFace) + V4.0 pure clustering + AMI
# Defaults: V4.0 calibrated values (best known configuration)

SPECIES_CONFIG = {
    "SalamanderID2025": {
        # V4.0 calibrated: mw=0.40 mgw=0.00 sw=0.40 kw=0.15 aw=0.05 thr=0.51
        "miew_weight": 0.40,
        "mega_weight": 0.00,
        "local_extractors": ["sift", "kaze", "aliked"],
        "local_weights": {"sift": 0.40, "kaze": 0.15, "aliked": 0.05},
        "threshold_cluster": 0.51,
        "image_size": 512,
        "qe_k": 3,
    },
    "SeaTurtleID2022": {
        # V4.0 calibrated: mw=0.50 mgw=0.30 sw=0.10 kw=0.10 aw=0.00 thr=0.65
        "miew_weight": 0.50,
        "mega_weight": 0.30,
        "local_extractors": ["sift", "kaze", "aliked"],
        "local_weights": {"sift": 0.10, "kaze": 0.10, "aliked": 0.00},
        "threshold_cluster": 0.65,
        "image_size": 512,
        "qe_k": 8,
    },
    "LynxID2025": {
        # V4.0 calibrated: mw=0.40 mgw=0.00 sw=0.05 kw=0.00 aw=0.55 thr=0.67
        "miew_weight": 0.40,
        "mega_weight": 0.00,
        "local_extractors": ["sift", "kaze", "aliked"],
        "local_weights": {"sift": 0.05, "kaze": 0.00, "aliked": 0.55},
        "threshold_cluster": 0.67,
        "image_size": 512,
        "qe_k": 5,
    },
    "TexasHornedLizards": {
        # V4.0 uncalibrated
        "miew_weight": 0.275,
        "mega_weight": 0.275,
        "local_extractors": ["sift", "kaze", "aliked"],
        "local_weights": {"sift": 0.15, "kaze": 0.10, "aliked": 0.20},
        "threshold_cluster": 0.30,
        "image_size": 512,
        "qe_k": 5,
    },
}

# Verify weights sum to 1.0
for species, cfg in SPECIES_CONFIG.items():
    total = cfg["miew_weight"] + cfg["mega_weight"] + sum(cfg["local_weights"].values())
    assert abs(total - 1.0) < 0.01, f"{species} weights don't sum to 1.0: {total}"

print("\u2713 Species configuration loaded (V5.1: fine-tuned MegaDescriptor + V4.0 defaults)")
for species, cfg in SPECIES_CONFIG.items():
    extractors_str = " + ".join(
        f"{k.upper()}={v:.0%}" for k, v in cfg["local_weights"].items()
    )
    print(f"  {species}: MiewID {cfg['miew_weight']:.0%}, MegaDesc {cfg['mega_weight']:.0%}, {extractors_str}")


In [ ]:
# Cell 1.4: Device Detection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 4
ROOT_DIR = "/kaggle/input/animal-clef-2026"
if not os.path.exists(ROOT_DIR):
    ROOT_DIR = "/kaggle/input/competitions/animal-clef-2026"
assert os.path.exists(ROOT_DIR), f"Competition data not found at {ROOT_DIR}"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    if torch.cuda.device_count() > 1:
        print(f"🚀 Using {torch.cuda.device_count()} GPUs") 

# Create cache directory
os.makedirs("cache/embeddings", exist_ok=True)
os.makedirs("cache/local_features", exist_ok=True)
os.makedirs("cache/match_scores", exist_ok=True)
print("✓ Cache directories created")

In [ ]:
# V3.0 Cell 1.4b: Preload cache from animalclef-v2-8-cache dataset
# ─────────────────────────────────────────────────────────────────────────────
# The animalclef-v2-8-cache dataset contains precomputed global embeddings
# and SIFT/KAZE local features for all 4 species, including Salamander with
# the V2.8 yellow+specular mask already applied.
# match_scores/ is intentionally skipped — V3.0 computes new
# *_ransac_matches.npy files with the FM_RANSAC geometric gate.

import shutil

# Kaggle mounts datasets at /kaggle/input/{slug}/; try both naming conventions
_V28_CANDIDATES = [
    "/kaggle/input/datasets/sreevaatsavbavana/animalclef-v2-8-cache/cache",
    "/kaggle/input/animalclef-v2-8-cache/cache",
]

_V28_SRC = None
for _p in _V28_CANDIDATES:
    if os.path.exists(_p):
        _V28_SRC = _p
        break

if _V28_SRC:
    print(f"V2.8 cache found: {_V28_SRC}")
    _copied = 0
    for _sub in ["embeddings", "local_features"]:
        _src_dir = os.path.join(_V28_SRC, _sub)
        _dst_dir = os.path.join("cache", _sub)
        os.makedirs(_dst_dir, exist_ok=True)
        if os.path.isdir(_src_dir):
            for _fname in sorted(os.listdir(_src_dir)):
                _src_f = os.path.join(_src_dir, _fname)
                _dst_f = os.path.join(_dst_dir, _fname)
                if not os.path.exists(_dst_f):
                    shutil.copy2(_src_f, _dst_f)
                    print(f"  Copied  {_sub}/{_fname}")
                    _copied += 1
                else:
                    print(f"  Present {_sub}/{_fname}")
    print(f"✓ V2.8 cache preloaded ({_copied} new files)")
else:
    print("⚠ V2.8 cache dataset not mounted — features will be extracted from scratch")

# ── V3 cache: embeddings + SIFT/KAZE local features + calib_cache ──────────────────
# What is reused: embeddings, SIFT/KAZE pkl, calib_cache (all 4 matrices per species).
# Skipped: *_aliked.pkl (ALIKEDExtractor now applies seg_mask → features changed).
# Skipped: match_scores/ (SIFT/KAZE suffix _ransac→_ratio; ALIKED features changed).
_V3_CANDIDATES = [
    "/kaggle/input/datasets/svrresearch/animalclef-v3-cache",
    "/kaggle/input/animalclef-v3-cache",
]
_V3_SRC = None
for _p in _V3_CANDIDATES:
    if os.path.exists(_p):
        _V3_SRC = _p
        break

if _V3_SRC:
    print(f"\nV3 cache found: {_V3_SRC}")
    _copied = 0

    # Embeddings (MiewID unchanged — always valid)
    _e_src = os.path.join(_V3_SRC, "cache", "embeddings")
    _e_dst = os.path.join("cache", "embeddings")
    os.makedirs(_e_dst, exist_ok=True)
    if os.path.isdir(_e_src):
        for _fname in sorted(os.listdir(_e_src)):
            _src_f = os.path.join(_e_src, _fname)
            _dst_f = os.path.join(_e_dst, _fname)
            if not os.path.exists(_dst_f):
                shutil.copy2(_src_f, _dst_f); _copied += 1
                print(f"  Copied  embeddings/{_fname}")
            else:
                print(f"  Present embeddings/{_fname}")

    # SIFT + KAZE local features (extractors unchanged)
    # Skip *_aliked.pkl — ALIKEDExtractor now applies seg_mask (features changed)
    _lf_src = os.path.join(_V3_SRC, "cache", "local_features")
    _lf_dst = os.path.join("cache", "local_features")
    os.makedirs(_lf_dst, exist_ok=True)
    if os.path.isdir(_lf_src):
        for _fname in sorted(os.listdir(_lf_src)):
            if "_aliked.pkl" in _fname:
                print(f"  Skipped local_features/{_fname}  (seg_mask fix → must recompute)")
                continue
            _src_f = os.path.join(_lf_src, _fname)
            _dst_f = os.path.join(_lf_dst, _fname)
            if not os.path.exists(_dst_f):
                shutil.copy2(_src_f, _dst_f); _copied += 1
                print(f"  Copied  local_features/{_fname}")
            else:
                print(f"  Present local_features/{_fname}")

    # calib_cache — ALL matrices valid (see comment above)
    _cc_src = os.path.join(_V3_SRC, "calib_cache")
    _cc_dst = "/kaggle/working/calib_cache"
    os.makedirs(_cc_dst, exist_ok=True)
    if os.path.isdir(_cc_src):
        for _fname in sorted(os.listdir(_cc_src)):
            _src_f = os.path.join(_cc_src, _fname)
            _dst_f = os.path.join(_cc_dst, _fname)
            if not os.path.exists(_dst_f):
                shutil.copy2(_src_f, _dst_f); _copied += 1
                print(f"  Copied  calib_cache/{_fname}")
            else:
                print(f"  Present calib_cache/{_fname}")

    print(f"✓ V3 cache preloaded ({_copied} new files)")
else:
    print("⚠ V3 cache not mounted — will recompute ALIKED features + all match matrices")


In [ ]:
# Cell 1.5: Load Data

print("📂 Loading AnimalCLEF 2026 dataset...")
full_dataset = AnimalCLEF2026(root=ROOT_DIR, transform=None, load_label=True)
metadata = full_dataset.metadata

test_meta = metadata[metadata["split"] == "test"]
train_meta = metadata[metadata["split"] == "train"]

print(f"✓ Dataset loaded")
print(f"  Total images: {len(metadata):,}")
print(f"  Test images: {len(test_meta):,}")
print(f"  Train images: {len(train_meta):,}")
print(f"\nSpecies breakdown:")
for species in test_meta["dataset"].unique():
    n_test = len(test_meta[test_meta["dataset"] == species])
    n_train = len(train_meta[train_meta["dataset"] == species])
    print(f"  {species}: {n_test} test, {n_train} train")

In [ ]:
# Cell 1.6: SAM 3 — build SEG_MAP and keypoint-mask helper
#
# Strategy (V1.3): run SIFT on the ORIGINAL image, then discard keypoints
# that land on background pixels.  This avoids the white-boundary artifacts
# that hurt V1.2 (score 0.26330 vs V1 baseline 0.30655).
#
# LynxID2025 has 0% cache coverage → get_seg_mask() returns None → no change
# for that species.  All other species get filtered keypoints.
from pathlib import Path
import cv2
import numpy as np

SEG_CACHE_ROOT = Path('/kaggle/input/datasets/sreevaatsavbavana/animalclef-26-sam3/sam3_yolo_output/segmented_images')
_root = Path(ROOT_DIR)  # ROOT_DIR defined in Cell 1.4 as a str

# Key: relative path  e.g. 'SeaTurtleID2022/img001.jpg'
SEG_MAP = {}  # {rel_key: Path_to_segmented_jpg}

all_meta = pd.concat([train_meta, test_meta])
for _, row in tqdm(all_meta.iterrows(), total=len(all_meta), desc='Building SEG_MAP'):
    stem    = Path(row['path']).stem
    ds_name = row['dataset']
    rel_key = str(row['path'])
    cached  = SEG_CACHE_ROOT / ds_name / (stem + '.jpg')
    if cached.exists():
        SEG_MAP[rel_key] = cached

n_total    = len(all_meta)
n_cached   = len(SEG_MAP)
n_fallback = n_total - n_cached
print(f'SEG_MAP built: {n_cached:,} cached  |  {n_fallback:,} fallback to original  |  {n_total:,} total')

# Per-species / per-split breakdown
print()
print(f'{"Dataset":<25} {"split":<6} {"cached":>7} {"total":>7} {"coverage":>9}')
print('-' * 58)
for ds in sorted(all_meta["dataset"].unique()):
    for split, meta_split in [("train", train_meta), ("test", test_meta)]:
        rows = meta_split[meta_split["dataset"] == ds]
        if len(rows) == 0:
            continue
        n_hit = sum(1 for _, r in rows.iterrows() if str(r["path"]) in SEG_MAP)
        pct   = 100 * n_hit / len(rows)
        flag  = "" if pct == 100 else " ⚠" if pct < 50 else " ✓"
        print(f'  {ds:<23} {split:<6} {n_hit:>7,} {len(rows):>7,} {pct:>8.1f}%{flag}')

def get_seg_mask(img_path):
    """
    Return a binary uint8 mask (1=animal, 0=background) derived from the
    pre-segmented image, or None if not in the cache.

    The segmented image has background pixels set to pure white (255,255,255).
    Mask = pixels that are NOT pure white.  Imperfect for white-furred/scaled
    animals, but good enough for SeaTurtles, Salamanders, and TexasLizards.
    """
    p = Path(img_path)
    try:
        rel_key = str(p.relative_to(_root))
    except ValueError:
        return None
    if rel_key not in SEG_MAP:
        return None
    seg = cv2.imread(str(SEG_MAP[rel_key]))
    if seg is None:
        return None
    # Background was painted to (255, 255, 255) — mark those as 0
    is_bg = (seg[:, :, 0] == 255) & (seg[:, :, 1] == 255) & (seg[:, :, 2] == 255)
    mask  = (~is_bg).astype(np.uint8)
    # ── V2.8: Salamander yellow-spot + specular filter ────────────────────
    # Flash photography creates specular hotspots (~27% of SIFT kpts) that
    # carry no identity information.  Yellow spots are the actual per-
    # individual marker on fire salamanders.
    if 'SalamanderID2025' in rel_key:
        orig = cv2.imread(str(p))
        if orig is not None:
            hsv      = cv2.cvtColor(orig, cv2.COLOR_BGR2HSV)
            specular = (hsv[:, :, 2] > 220) & (hsv[:, :, 1] < 40)
            yellow   = ((hsv[:, :, 0] >= 18) & (hsv[:, :, 0] <= 42) &
                        (hsv[:, :, 1] >  80) & (hsv[:, :, 2] >  80))
            _k       = np.ones((25, 25), np.uint8)
            yellow_d = cv2.dilate(yellow.astype(np.uint8), _k).astype(bool)
            no_spec  = mask.astype(bool) & ~specular
            mask     = (no_spec & yellow_d).astype(np.uint8) if yellow_d.sum() >= 50 \
                       else no_spec.astype(np.uint8)
    # ─────────────────────────────────────────────────────────────────────
    return mask

print('get_seg_mask() ready.')


## Section 2: Global Features (MiewID v3 + Fine-tuned MegaDescriptor)

In [ ]:
# Cell 2.1: MiewID v3 Model

class MiewIDWrapper(nn.Module):
    """MiewID v3 wrapper for global feature extraction."""
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            "conservationxlabs/miewid-msv3",
            trust_remote_code=True
        )

    def forward(self, x):
        return self.backbone(x)


def get_global_model():
    """Initialize MiewID v3 model with multi-GPU support."""
    model = MiewIDWrapper()
    model.to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    model.eval()
    return model

print("✓ MiewID v3 model wrapper defined")

In [ ]:
# Cell 2.2: Extract Global Features

def extract_global_features(model, dataset, image_size):
    """Extract L2-normalized global features with TTA (horizontal flip)."""
    dataset.transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    
    loader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        num_workers=NUM_WORKERS, 
        shuffle=False
    )
    
    all_features = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting global features", leave=False):
            images = batch[0].to(DEVICE)
            
            # TTA: original + horizontal flip
            with torch.cuda.amp.autocast():
                feats_sum = model(images) + model(torch.flip(images, dims=[3]))
            
            # L2 normalize
            feats_norm = torch.nn.functional.normalize(feats_sum.float(), p=2, dim=1)
            all_features.append(feats_norm.cpu().numpy())
    
    return np.concatenate(all_features)

print("✓ Global feature extraction function defined")

In [ ]:
# Cell 2.3: Cache Global Embeddings

global_features_cache = {}
model = get_global_model()

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Global Features")
    
    cfg = SPECIES_CONFIG[species]
    cache_file = f"cache/embeddings/{species}_global.npy"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached embeddings: {cache_file}")
        features = np.load(cache_file)
    else:
        # Extract features
        sp_meta = test_meta[test_meta["dataset"] == species]
        sp_dataset = full_dataset.get_subset(sp_meta.index.values)
        
        features = extract_global_features(model, sp_dataset, cfg["image_size"])
        
        # Cache to disk
        np.save(cache_file, features)
        print(f"  Cached to {cache_file}")
    
    global_features_cache[species] = features
    print(f"  Shape: {features.shape}, Norm: {np.linalg.norm(features[0]):.3f}")

del model
torch.cuda.empty_cache()
print("\n✓ Global features extracted and cached")

In [ ]:
# Cell 2.3b: Fine-tuned MegaDescriptor Model + Extraction

# V5.1: Per-species model selection:
#   - Species WITH fine-tuned checkpoint: MegaDescriptor-T-224 + loaded ArcFace weights
#   - Species WITHOUT checkpoint (e.g., THL): pretrained MegaDescriptor-L-384 (V4.0 behavior)

FINETUNE_DIR = Path("/kaggle/input/megadesc-finetuned-v5")
if not FINETUNE_DIR.exists():
    FINETUNE_DIR = Path("/kaggle/input/datasets/sreevaatsavbavana/megadesc-finetuned-v5")
FINETUNE_SPECIES_FILES = {
    "LynxID2025":       "LynxID2025_megadesc_v5.pth",
    "SalamanderID2025": "SalamanderID2025_megadesc_v5.pth",
    "SeaTurtleID2022":  "SeaTurtleID2022_megadesc_v5.pth",
}

# Detect if any fine-tuned checkpoints exist
_ft_available = any((FINETUNE_DIR / f).exists() for f in FINETUNE_SPECIES_FILES.values())
if _ft_available:
    print("\u2713 Fine-tuned MegaDescriptor-T-224 checkpoints detected")
    for sp, fname in FINETUNE_SPECIES_FILES.items():
        exists = (FINETUNE_DIR / fname).exists()
        print(f"  {sp}: {'\u2713' if exists else '\u2717 (will use L-384 pretrained)'} {fname}")
    print("  Species not in FINETUNE_SPECIES_FILES use pretrained L-384 fallback")
else:
    print("\u26a0 No fine-tuned checkpoints found -- all species use pretrained MegaDescriptor-L-384")

# Cache for loaded models (avoid reloading per species)
_mega_models_cache = {}


def _species_has_ft(species):
    '''Check if a species has a fine-tuned checkpoint available.'''
    if not _ft_available or not species:
        return False
    fname = FINETUNE_SPECIES_FILES.get(species)
    return bool(fname and (FINETUNE_DIR / fname).exists())


def get_mega_config(species=None):
    '''Return (model_name, image_size, batch_size) for this species MegaDescriptor variant.'''
    if _species_has_ft(species):
        return 'hf-hub:BVRA/MegaDescriptor-T-224', 224, 64
    else:
        return 'hf-hub:BVRA/MegaDescriptor-L-384', 384, 32


def get_mega_model(species=None):
    '''Load MegaDescriptor model. Fine-tuned T-224 if available, else pretrained L-384.'''
    ft_path = None
    if _species_has_ft(species):
        fname = FINETUNE_SPECIES_FILES[species]
        ft_path = FINETUNE_DIR / fname

    model_name, _, _ = get_mega_config(species)
    cache_key = str(ft_path) if ft_path else f"pretrained_{model_name}"

    if cache_key in _mega_models_cache:
        return _mega_models_cache[cache_key]

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=0,  # CRITICAL: without this, returns class logits not embeddings
    )

    if ft_path:
        print(f"  Loading fine-tuned weights: {ft_path.name}")
        state = torch.load(str(ft_path), map_location="cpu")
        model.load_state_dict(state)
    else:
        print(f"  Using pretrained {model_name} for {species}")

    model = model.eval().to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    _mega_models_cache[cache_key] = model
    return model


def extract_mega_features(model, dataset, image_size, batch_size=32):
    '''Extract L2-normalized MegaDescriptor features with TTA (horizontal flip).'''
    dataset.transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        shuffle=False,
    )

    all_features = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting MegaDescriptor features", leave=False):
            images = batch[0].to(DEVICE)
            with torch.cuda.amp.autocast():
                feats = model(images) + model(torch.flip(images, dims=[3]))
            feats_norm = torch.nn.functional.normalize(feats.float(), p=2, dim=1)
            all_features.append(feats_norm.cpu().numpy())

    return np.concatenate(all_features)


print("\u2713 MegaDescriptor model + extraction function defined (V5.1)")


In [ ]:
# Cell 2.4: Query Expansion (Optional)

def query_expansion(features, k=5, alpha=0.5):
    """Query expansion via k-nearest neighbors averaging."""
    sim_matrix = np.dot(features, features.T)
    knn_indices = np.argsort(sim_matrix, axis=1)[:, -k:]
    
    expanded = np.zeros_like(features)
    for i in range(features.shape[0]):
        expanded[i] = features[i] + alpha * np.mean(features[knn_indices[i]], axis=0)
    
    return normalize(expanded, axis=1, norm="l2")


# Apply query expansion
global_features_expanded = {}
for species, features in global_features_cache.items():
    cfg = SPECIES_CONFIG[species]
    expanded = query_expansion(features, k=cfg["qe_k"])
    global_features_expanded[species] = expanded
    print(f"{species}: QE with k={cfg['qe_k']}")

print("✓ Query expansion applied to global features")

In [ ]:
# Cell 2.3c: Cache MegaDescriptor Embeddings + Query Expansion

# V5.1: Per-species model selection.
#   Fine-tuned species: T-224, cache suffix _mega_ft
#   Other species (THL): pretrained L-384, cache suffix _mega

mega_features_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - MegaDescriptor Features")

    mega_model = get_mega_model(species=species)
    _, mega_img_size, mega_batch = get_mega_config(species)
    _is_ft = _species_has_ft(species)

    cache_suffix = "_mega_ft" if _is_ft else "_mega"
    cache_file = f"cache/embeddings/{species}{cache_suffix}.npy"

    if os.path.exists(cache_file):
        print(f"  Loading cached embeddings: {cache_file}")
        features = np.load(cache_file)
    else:
        sp_meta = test_meta[test_meta["dataset"] == species]
        sp_dataset = full_dataset.get_subset(sp_meta.index.values)
        features = extract_mega_features(mega_model, sp_dataset, image_size=mega_img_size, batch_size=mega_batch)
        np.save(cache_file, features)
        print(f"  Cached to {cache_file}")

    mega_features_cache[species] = features
    print(f"  Shape: {features.shape}, Norm: {np.linalg.norm(features[0]):.3f}")
    print(f"  Model: {'fine-tuned T-224' if _is_ft else 'pretrained L-384'}")

# Clean up model cache to free GPU memory
_mega_models_cache.clear()
torch.cuda.empty_cache()
print("\n\u2713 MegaDescriptor features extracted and cached")

# Apply query expansion to MegaDescriptor (same QE as MiewID)
mega_features_expanded = {}
for species, features in mega_features_cache.items():
    cfg = SPECIES_CONFIG[species]
    expanded = query_expansion(features, k=cfg["qe_k"])
    mega_features_expanded[species] = expanded
    print(f"{species}: MegaDesc QE with k={cfg['qe_k']}")

print("\u2713 Query expansion applied to MegaDescriptor features")


## Section 3: Local Features (SIFT, KAZE, ALIKED)

In [ ]:
# Cell 3.1: SIFT Extractor

class SIFTExtractor:
    """SIFT feature extractor using OpenCV (rotation-invariant)."""
    def __init__(self, max_keypoints=1000):
        self.max_keypoints = max_keypoints
        self.sift = cv2.SIFT_create(nfeatures=max_keypoints)
    
    def extract(self, image_path):
        """Extract SIFT keypoints and descriptors."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Convert to grayscale for SIFT
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Detect and compute
        keypoints, descriptors = self.sift.detectAndCompute(gray, None)
        
        if descriptors is None or len(keypoints) < 4:
            return None
        
        # RootSIFT: L1-normalize then element-wise sqrt (HotSpotter's descriptor).
        # More discriminative for texture-heavy patterns (turtles, lizards).
        # SIFT descriptors are non-negative (gradient-magnitude histograms).
        descriptors /= (descriptors.sum(axis=1, keepdims=True) + 1e-7)
        descriptors = np.sqrt(descriptors)
        
        # Filter keypoints to animal region using segmentation mask.
        # SIFT runs on the ORIGINAL image (no white-bg boundary artifacts).
        # get_seg_mask() returns None for uncached images → all kpts kept.
        seg_mask = get_seg_mask(image_path)
        if seg_mask is not None:
            h, w = seg_mask.shape
            keep = [
                i for i, kp in enumerate(keypoints)
                if 0 <= int(kp.pt[1]) < h
                and 0 <= int(kp.pt[0]) < w
                and seg_mask[int(kp.pt[1]), int(kp.pt[0])] == 1
            ]
            if len(keep) >= 4:
                keypoints   = [keypoints[i] for i in keep]
                descriptors = descriptors[keep]
        
        # Convert keypoints to numpy array (x, y)
        kpts_array = np.array([kp.pt for kp in keypoints], dtype=np.float32)
        scores_array = np.array([kp.response for kp in keypoints], dtype=np.float32)
        
        return {
            'keypoints': kpts_array,
            'descriptors': descriptors.astype(np.float32),
            'scores': scores_array
        }

print("✓ SIFT extractor defined")

In [ ]:
# Cell 3.2: Feature Extractor Strategy

# SIMPLIFIED APPROACH: SIFT-Only Ensemble
# 
# Due to kornia compatibility issues (ALIKED/DISK not available in all versions),
# we use a proven, robust approach:
#
# 1. SIFT (OpenCV): Classical rotation-invariant features
#    - Works on all platforms
#    - 128-dim descriptors
#    - Matched with BFMatcher + Lowe's ratio test
#    - Excellent for rotation/scale invariance
#
# 2. Global (MiewID v3): Deep learning embeddings
#    - 2152-dim features
#    - Pre-trained on wildlife data
#    - Excellent for overall similarity
#
# Ensemble: Global (65-70%) + SIFT (30-35%)
# This provides a good balance of deep learning and classical CV approaches

print("✓ Using dual-global (MiewID v3 + fine-tuned MegaDescriptor) + SIFT/KAZE/ALIKED ensemble (V5.1)")

In [ ]:
# Cell 3.3: ALIKED Extractor (LightGlue)

from lightglue import ALIKED as _ALIKED_Extractor
from lightglue.utils import load_image as _load_aliked_img

class ALIKEDExtractor:
    """ALIKED feature extractor via LightGlue (deformation-robust, 128-dim)."""
    def __init__(self, max_keypoints=1024, device='cuda'):
        self.device = device
        self.model = _ALIKED_Extractor(max_num_keypoints=max_keypoints).eval().to(device)

    def extract(self, image_path):
        """Extract ALIKED keypoints and descriptors."""
        try:
            img = _load_aliked_img(str(image_path)).to(self.device)
            with torch.no_grad():
                feats = self.model.extract(img)
            kps     = feats['keypoints'][0].cpu().numpy()
            descs   = feats['descriptors'][0].cpu().numpy()
            scores  = feats['keypoint_scores'][0].cpu().numpy()
            img_sz  = feats['image_size'][0].cpu().numpy()
            # Filter keypoints to animal region — matches _extract_calib_aliked()
            seg_mask = get_seg_mask(str(image_path))
            if seg_mask is not None:
                h, w = seg_mask.shape[:2]
                xs = np.clip(kps[:, 0].astype(int), 0, w - 1)
                ys = np.clip(kps[:, 1].astype(int), 0, h - 1)
                valid  = seg_mask[ys, xs] > 0
                kps    = kps[valid]
                descs  = descs[valid]
                scores = scores[valid]
            if len(kps) < 4:
                return None
            return {
                'keypoints': kps,
                'descriptors': descs,
                'scores': scores,
                'image_size': img_sz,
            }
        except Exception:
            return None

print("✓ ALIKED extractor defined (LightGlue, seg_mask filtered)")

In [ ]:
# Cell 3.4: KAZE Extractor (OpenCV)
#
# KAZE uses a non-linear (anisotropic diffusion) scale space instead of the
# Gaussian pyramid used by SIFT. This gives keypoints that are more stable
# near object boundaries and textured regions.
#
# Advantages over ALIKED/DISK for this environment:
#   - Pure OpenCV: no kornia, no pretrained weights, no internet needed
#   - Float descriptors (64-dim): compatible with existing BFMatcher + L2
#   - SAM3 mask filter applied (SeaTurtle has 100% coverage)
#   - RootSIFT does NOT apply: KAZE uses M-SURF-style descriptors with signed
#     Haar wavelet responses (sum_dx, sum_dy can be negative). sqrt(negative) = NaN.
#
# Note: KAZE has no built-in max_features parameter. We sort by keypoint
# response and keep the top max_keypoints after detection.

class KAZEExtractor:
    """KAZE feature extractor using OpenCV (non-linear scale space, float descriptors)."""
    def __init__(self, max_keypoints=1000):
        self.max_keypoints = max_keypoints
        self.kaze = cv2.KAZE_create(upright=False)  # upright=False = rotation invariant

    def extract(self, image_path):
        """Extract KAZE keypoints and descriptors with SAM3 mask filter (no RootSIFT)."""
        img = cv2.imread(str(image_path))
        if img is None:
            return None

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptors = self.kaze.detectAndCompute(gray, None)

        if descriptors is None or len(keypoints) < 4:
            return None

        # Limit to max_keypoints by response strength (KAZE has no nfeatures param)
        if len(keypoints) > self.max_keypoints:
            sorted_pairs = sorted(enumerate(keypoints), key=lambda x: -x[1].response)
            keep_idx = sorted([i for i, _ in sorted_pairs[:self.max_keypoints]])
            keypoints   = [keypoints[i] for i in keep_idx]
            descriptors = descriptors[keep_idx]

        # NOTE: RootSIFT (L1-norm + sqrt) is NOT applied to KAZE descriptors.
        # KAZE uses an M-SURF-style descriptor that stores signed Haar wavelet
        # responses (sum_dx, sum_dy can be negative). Applying sqrt to negative
        # values produces NaN → silent zero-match failure.
        # KAZE descriptors are L2-normalised internally by OpenCV; use as-is.
        descriptors = descriptors.astype(np.float32)

        # SAM3 mask filter: discard keypoints outside the animal region.
        # SeaTurtleID2022 has 100% cache coverage → every image benefits.
        seg_mask = get_seg_mask(image_path)
        if seg_mask is not None:
            h, w = seg_mask.shape
            keep = [
                i for i, kp in enumerate(keypoints)
                if 0 <= int(kp.pt[1]) < h
                and 0 <= int(kp.pt[0]) < w
                and seg_mask[int(kp.pt[1]), int(kp.pt[0])] == 1
            ]
            if len(keep) >= 4:
                keypoints   = [keypoints[i] for i in keep]
                descriptors = descriptors[keep]

        kpts_array   = np.array([kp.pt       for kp in keypoints], dtype=np.float32)
        scores_array = np.array([kp.response for kp in keypoints], dtype=np.float32)

        return {
            'keypoints':   kpts_array,
            'descriptors': descriptors.astype(np.float32),
            'scores':      scores_array,
        }

print("✓ KAZE extractor defined (OpenCV, no pretrained weights, no RootSIFT)")


In [ ]:
# Cell 3.5: Extract Local Features per Species

def get_extractor(extractor_name, device='cuda'):
    """Factory function to create feature extractors."""
    if extractor_name == 'sift':
        return SIFTExtractor(max_keypoints=1000)
    elif extractor_name == 'kaze':
        return KAZEExtractor(max_keypoints=1000)
    elif extractor_name == 'aliked':
        return ALIKEDExtractor(max_keypoints=1024, device=device)
    else:
        raise ValueError(f"Unknown extractor: {extractor_name}. Supported: 'sift', 'kaze', 'aliked'.")


def extract_local_features_for_dataset(species, extractor_name, dataset_meta, root_dir):
    """Extract local features for all images in a dataset split."""
    cache_file = f"cache/local_features/{species}_{extractor_name}.pkl"
    
    # Check cache
    if os.path.exists(cache_file):
        print(f"  Loading cached {extractor_name} features: {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Extract features
    print(f"  Extracting {extractor_name} features for {len(dataset_meta)} images...")
    extractor = get_extractor(extractor_name, DEVICE)
    
    features_list = []
    for idx, row in tqdm(dataset_meta.iterrows(), total=len(dataset_meta), leave=False, desc=f"{extractor_name}"):
        img_path = os.path.join(root_dir, row['path'])
        feats = extractor.extract(img_path)
        features_list.append(feats)
    
    # Cache to disk
    with open(cache_file, 'wb') as f:
        pickle.dump(features_list, f)
    print(f"  Cached to {cache_file}")
    
    return features_list


# Extract local features for each species
local_features_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Local Features (SIFT)")
    
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    local_features_cache[species] = {}
    for extractor_name in cfg["local_extractors"]:
        features = extract_local_features_for_dataset(
            species, extractor_name, sp_meta, ROOT_DIR
        )
        local_features_cache[species][extractor_name] = features
        
        # Stats
        valid_count = sum(1 for f in features if f is not None)
        if valid_count > 0:
            avg_kpts = np.mean([len(f['keypoints']) for f in features if f is not None])
            print(f"    ✓ {extractor_name}: {valid_count}/{len(features)} valid, "
                  f"avg {avg_kpts:.0f} keypoints")
        else:
            print(f"    ⚠ {extractor_name}: No valid features extracted!")

print("\n✓ Local features extracted and cached")

In [ ]:
# Cell 3.6: Verify Local Features Cache

print("Local Features Summary:")
for species, extractors_dict in local_features_cache.items():
    print(f"\n{species}:")
    for extractor_name, features_list in extractors_dict.items():
        valid = sum(1 for f in features_list if f is not None)
        print(f"  {extractor_name}: {valid}/{len(features_list)} images with features")

print("\n✓ Local features verified")

## Section 4: Matching with LightGlue

In [ ]:
# Cell 4.1: SIFT Matcher (BFMatcher)

class SIFTMatcher:
    """SIFT matcher using BFMatcher with Lowe's ratio test."""
    def __init__(self, device='cuda'):
        # BFMatcher for SIFT (L2 norm)
        self.matcher = cv2.BFMatcher(cv2.NORM_L2)
    
    def match(self, feat0, feat1):
        """Match SIFT keypoints between two images."""
        if feat0 is None or feat1 is None:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        if len(feat0['descriptors']) < 2 or len(feat1['descriptors']) < 2:
            return {'matches': np.array([]), 'match_indices': np.array([]), 'confidence': np.array([])}
        
        # kNN matching with k=2
        matches = self.matcher.knnMatch(feat0['descriptors'], feat1['descriptors'], k=2)
        
        # Lowe's ratio test (0.75 threshold)
        good_matches = []
        for pair in matches:
            if len(pair) == 2:
                m, n = pair
                if m.distance < 0.75 * n.distance:
                    good_matches.append(m)
        
        if len(good_matches) == 0:
            return {
                'matches': np.array([], dtype=int),
                'match_indices': np.array([], dtype=int),
                'confidence': np.array([], dtype=np.float32)
            }
        
        return {
            'matches': np.array([m.queryIdx for m in good_matches], dtype=int),
            'match_indices': np.array([m.trainIdx for m in good_matches], dtype=int),
            'confidence': 1.0 - np.array([m.distance for m in good_matches], dtype=np.float32) / 255.0
        }

print("✓ SIFT matcher defined (BFMatcher with Lowe's ratio test)")

In [ ]:
# Cell 4.2: Ultra-Fast GPU Batch Matching

def batch_sift_match_gpu(desc1, desc2, ratio_thresh=0.75):
    """GPU-accelerated SIFT matching using PyTorch batch operations."""
    if desc1 is None or desc2 is None or len(desc1) < 2 or len(desc2) < 2:
        return 0
    
    # Convert to torch tensors on GPU
    d1 = torch.from_numpy(desc1).float().to(DEVICE)
    d2 = torch.from_numpy(desc2).float().to(DEVICE)
    
    # Compute pairwise L2 distances (batch operation on GPU)
    # distances[i, j] = L2 distance between d1[i] and d2[j]
    distances = torch.cdist(d1, d2, p=2)  # Shape: (N1, N2)
    
    # Get 2 nearest neighbors for each descriptor in d1
    topk_dists, topk_indices = torch.topk(distances, k=min(2, d2.shape[0]), 
                                          dim=1, largest=False)
    
    # Lowe's ratio test (vectorized)
    if topk_dists.shape[1] == 2:
        ratios = topk_dists[:, 0] / (topk_dists[:, 1] + 1e-8)
        good_matches = (ratios < ratio_thresh).sum().item()
    else:
        good_matches = topk_dists.shape[0]  # All are good if only 1 neighbor
    
    return good_matches


# ── V3.0: GPU ratio test + CPU FM_RANSAC geometric gate ───────────────────────

MAX_RANSAC_PTS = 100  # cap for cv2 FM_RANSAC — ~2ms/pair at this size

def compute_local_score_gpu(feat1, feat2, ratio_thresh=0.75):
    """GPU ratio test + CPU FM_RANSAC geometric gate.

    Stage 1 (GPU): torch.cdist + Lowe ratio test.
    Stage 2 (CPU): cv2.FM_RANSAC on ≤MAX_RANSAC_PTS matched points.
      If n_inliers < 4: return 0.0  (geometrically inconsistent — reject).
      Else: return 1 - exp(-n_ratio_test_matches / 20).

    Score scale is consistent with calibration: _calib_match_matrix uses
    BFMatcher good-match counts in 1-exp(-n/20), and n_ratio_test ≈ n_bfm_good.
    RANSAC acts only as a binary gate — it does not change the score formula.
    """
    if feat1 is None or feat2 is None:
        return 0.0

    desc1 = torch.from_numpy(feat1['descriptors']).float().to(DEVICE)
    desc2 = torch.from_numpy(feat2['descriptors']).float().to(DEVICE)
    n1, n2 = len(desc1), len(desc2)
    if n1 < 8 or n2 < 8:
        return 0.0

    # Step 1: GPU pairwise distances + Lowe's ratio test
    dists = torch.cdist(desc1, desc2, p=2)
    k = min(2, n2)
    topk_dists, topk_idx = torch.topk(dists, k=k, dim=1, largest=False)

    if k == 2:
        ratios = topk_dists[:, 0] / (topk_dists[:, 1] + 1e-8)
        gmask  = ratios < ratio_thresh
        qidx   = torch.where(gmask)[0]
        tidx   = topk_idx[gmask, 0]
    else:
        qidx = torch.arange(n1, device=DEVICE)
        tidx = topk_idx[:, 0]

    n_matches = len(qidx)
    if n_matches < 8:
        return 0.0

    # Step 2: CPU FM_RANSAC — binary geometric gate only
    kp1  = torch.from_numpy(feat1['keypoints']).float()  # stay CPU
    kp2  = torch.from_numpy(feat2['keypoints']).float()
    pts1 = kp1[qidx.cpu()].numpy()   # (K, 2)
    pts2 = kp2[tidx.cpu()].numpy()   # (K, 2)

    # Cap for speed — random subsample is fine (geometric test, not score)
    if len(pts1) > MAX_RANSAC_PTS:
        rng  = np.random.default_rng(0)
        sel  = rng.choice(len(pts1), MAX_RANSAC_PTS, replace=False)
        pts1, pts2 = pts1[sel], pts2[sel]

    try:
        _, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC, 3.0, 0.99)
        n_inliers = int(mask.sum()) if mask is not None else 0
    except Exception:
        n_inliers = n_matches  # degenerate (collinear etc.): accept

    if n_inliers < 4:
        return 0.0

    # Score on ratio-test count — same scale as calibration
    return float(1.0 - np.exp(-n_matches / 20.0))


# ── V3.1: LightGlue matcher for ALIKED features ─────────────────────────────────

_aliked_lg_matcher = None

def _get_aliked_matcher():
    """Lazy-init the LightGlue matcher for ALIKED (loaded on first call)."""
    global _aliked_lg_matcher
    if _aliked_lg_matcher is None:
        from lightglue import LightGlue as _LG
        _aliked_lg_matcher = _LG(features='aliked').eval().to(DEVICE)
    return _aliked_lg_matcher


def compute_local_score_lightglue(feat1, feat2):
    """Score two images using ALIKED features matched via LightGlue.

    Returns: 1 - exp(-n_matches / 20) for consistency with SIFT/KAZE scoring.
    """
    if feat1 is None or feat2 is None:
        return 0.0
    if len(feat1['keypoints']) < 4 or len(feat2['keypoints']) < 4:
        return 0.0

    matcher = _get_aliked_matcher()

    d0 = {
        'keypoints': torch.from_numpy(feat1['keypoints']).float().unsqueeze(0).to(DEVICE),
        'descriptors': torch.from_numpy(feat1['descriptors']).float().unsqueeze(0).to(DEVICE),
    }
    d1 = {
        'keypoints': torch.from_numpy(feat2['keypoints']).float().unsqueeze(0).to(DEVICE),
        'descriptors': torch.from_numpy(feat2['descriptors']).float().unsqueeze(0).to(DEVICE),
    }
    if 'image_size' in feat1:
        d0['image_size'] = torch.tensor(feat1['image_size']).float().unsqueeze(0).to(DEVICE)
    if 'image_size' in feat2:
        d1['image_size'] = torch.tensor(feat2['image_size']).float().unsqueeze(0).to(DEVICE)

    try:
        with torch.no_grad():
            result = matcher({'image0': d0, 'image1': d1})
        n_matches = (result['matches0'][0] > -1).sum().item()
    except Exception:
        return 0.0

    return float(1.0 - np.exp(-n_matches / 20.0))


# ── V3.2: ratio-test-only scoring for SIFT/KAZE (no RANSAC) ────────────────

def compute_local_score_ratio_only(feat1, feat2, ratio_thresh=0.75):
    """GPU Lowe ratio test only — no RANSAC geometric gate.

    V3.2 fix: Calibration (_calib_match_matrix) uses GPU torch.cdist + ratio
    test without RANSAC. V3.1 used compute_local_score_gpu at test time which
    applies FM_RANSAC, making test scores systematically lower than calibration
    scores. Grid-search then incorrectly concluded kw=0 for all species.

    This function matches the calibration's score distribution exactly:
    Score = 1 − exp(−n_ratio_matches / 20), same formula as _calib_match_matrix.
    """
    if feat1 is None or feat2 is None:
        return 0.0
    desc1 = torch.from_numpy(feat1['descriptors']).float().to(DEVICE)
    desc2 = torch.from_numpy(feat2['descriptors']).float().to(DEVICE)
    n1, n2 = len(desc1), len(desc2)
    if n1 < 2 or n2 < 2:
        return 0.0

    dists = torch.cdist(desc1, desc2, p=2)
    k = min(2, n2)
    topk_dists, _ = torch.topk(dists, k=k, dim=1, largest=False)
    if k == 2:
        ratios    = topk_dists[:, 0] / (topk_dists[:, 1] + 1e-8)
        n_matches = int((ratios < ratio_thresh).sum().item())
    else:
        n_matches = n1
    return float(1.0 - np.exp(-n_matches / 20.0))


def compute_pairwise_matches_fast(features_list, extractor_type, species):
    """Ultra-fast pairwise matching using GPU batch operations."""
    n = len(features_list)
    _sfx = "_lg_matches" if extractor_type == "aliked" else "_ratio_matches"
    cache_file = f"cache/match_scores/{species}_{extractor_type}{_sfx}.npy"
    
    # Check cache FIRST
    if os.path.exists(cache_file):
        print(f"  ✓ Loading cached {extractor_type} match scores: {cache_file}")
        return np.load(cache_file)
    
    print(f"  Computing {extractor_type} matches for {n} images (GPU batch mode)...")
    
    # Top-K strategy: only match to most similar candidates
    K = min(50, n)
    
    # Get global similarity for candidate selection
    global_feats = global_features_expanded[species]
    global_sim = np.dot(global_feats, global_feats.T)
    
    # Pre-compute top-K candidates for ALL images at once
    top_k_all = np.argsort(global_sim, axis=1)[:, -K:]  # Shape: (N, K)
    
    # Initialize match matrix
    match_matrix = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(match_matrix, 1.0)
    
    # Process in batches for GPU efficiency
    batch_size = 50  # Process 50 query images at once
    
    for start_idx in tqdm(range(0, n, batch_size), desc=f"Matching {extractor_type}"):
        end_idx = min(start_idx + batch_size, n)
        
        # For each query in this batch
        for i in range(start_idx, end_idx):
            if features_list[i] is None:
                continue
            
            # Get top-K candidates for this query
            candidates = top_k_all[i]
            
            # Batch match against all candidates (GPU accelerated)
            for j in candidates:
                if i == j or match_matrix[i, j] > 0:
                    continue
                
                if features_list[j] is None:
                    continue
                
                # V3.2: LightGlue for ALIKED; ratio-test-only for SIFT/KAZE (no RANSAC)
                if extractor_type == 'aliked':
                    score = compute_local_score_lightglue(features_list[i], features_list[j])
                else:
                    # No RANSAC gate: score distribution now matches _calib_match_matrix
                    score = compute_local_score_ratio_only(features_list[i], features_list[j])
                
                match_matrix[i, j] = score
                match_matrix[j, i] = score
    
    # Cache to disk
    np.save(cache_file, match_matrix)
    print(f"  ✓ Cached to {cache_file}")
    
    return match_matrix

print("✓ Ultra-fast GPU batch matching defined")

In [ ]:
# Cell 4.3: RANSAC Verification and Match Scoring

def ransac_filter(kpts0, kpts1, matches, threshold=3.0):
    """RANSAC geometric verification of matches."""
    if len(matches['matches']) < 4:
        return np.array([], dtype=int)
    
    pts0 = kpts0[matches['matches']]
    pts1 = kpts1[matches['match_indices']]
    
    # Estimate fundamental matrix with RANSAC
    _, mask = cv2.findFundamentalMat(
        pts0, pts1, cv2.FM_RANSAC, threshold, confidence=0.99
    )
    
    if mask is None:
        return np.array([], dtype=int)
    
    return np.where(mask.ravel())[0]


def score_local_match(matches, feat0, feat1):
    """Convert matches to similarity score [0, 1]."""
    if feat0 is None or feat1 is None:
        return 0.0
    
    if len(matches['matches']) == 0:
        return 0.0
    
    # RANSAC geometric verification
    inlier_indices = ransac_filter(
        feat0['keypoints'], 
        feat1['keypoints'], 
        matches
    )
    
    num_inliers = len(inlier_indices)
    if num_inliers == 0:
        return 0.0
    
    # Average confidence of inlier matches
    avg_confidence = np.mean(matches['confidence'][inlier_indices])
    
    # Sigmoid-like normalization (plateaus at ~50 matches)
    match_score = 1.0 - np.exp(-num_inliers / 20.0)
    
    # Combine match count and confidence
    final_score = match_score * avg_confidence
    
    return float(final_score)

print("✓ RANSAC verification and match scoring functions defined")

In [ ]:
# Cell 4.4: Compute and Cache Match Scores

match_scores_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\n{'='*60}")
    print(f"Processing {species} - Match Scores")
    
    cfg = SPECIES_CONFIG[species]
    match_scores_cache[species] = {}
    
    for extractor_name in cfg["local_extractors"]:
        features_list = local_features_cache[species][extractor_name]
        
        # Use fast top-K matching strategy
        match_matrix = compute_pairwise_matches_fast(features_list, extractor_name, species)
        match_scores_cache[species][extractor_name] = match_matrix
        
        # Stats
        non_diag = match_matrix[~np.eye(match_matrix.shape[0], dtype=bool)]
        print(f"    ✓ {extractor_name}: Shape {match_matrix.shape}, "
              f"Mean score: {non_diag.mean():.3f}, "
              f"Non-zero pairs: {(non_diag > 0).sum():,}")

print("\n✓ Match scores computed and cached")

## Section 5: Ensemble Voting

In [ ]:
# Cell 5.1: Ensemble Scoring Function

def compute_ensemble_similarity_matrix(species):
    """Compute weighted ensemble similarity matrix (V4.0: dual-global)."""
    cfg = SPECIES_CONFIG[species]

    # Dual global similarity (SEPARATE matrices, not concatenated)
    miew_feats = global_features_expanded[species]
    miew_sim   = np.dot(miew_feats, miew_feats.T)

    mega_feats = mega_features_expanded[species]
    mega_sim   = np.dot(mega_feats, mega_feats.T)

    # Weighted dual-global
    ensemble_sim = cfg["miew_weight"] * miew_sim + cfg["mega_weight"] * mega_sim

    # Add weighted local match scores
    for extractor_name, weight in cfg["local_weights"].items():
        local_sim = match_scores_cache[species][extractor_name]
        ensemble_sim += weight * local_sim

    return ensemble_sim

print("✓ Ensemble scoring function defined (V4.0: dual-global)")


In [ ]:
# Cell 5.2: Compute Ensemble Scores for All Species

ensemble_similarity_cache = {}

for species in test_meta["dataset"].unique():
    print(f"\nComputing ensemble scores for {species}...")
    ensemble_sim = compute_ensemble_similarity_matrix(species)
    ensemble_similarity_cache[species] = ensemble_sim
    
    # Stats
    print(f"  Shape: {ensemble_sim.shape}")
    print(f"  Mean similarity: {ensemble_sim.mean():.3f}")
    print(f"  Std similarity: {ensemble_sim.std():.3f}")
    print(f"  Min/Max: {ensemble_sim.min():.3f} / {ensemble_sim.max():.3f}")

print("\n✓ Ensemble similarity matrices computed")

In [ ]:
# Cell 5.3: Weighted Voting Summary

print("Ensemble Weights Summary (V4.0: dual-global):\n")
for species, cfg in SPECIES_CONFIG.items():
    print(f"{species}:")
    print(f"  MiewID v3: {cfg['miew_weight']:.0%}")
    print(f"  MegaDescriptor-L: {cfg['mega_weight']:.0%}")
    for extractor, weight in cfg['local_weights'].items():
        print(f"  {extractor.upper()}: {weight:.0%}")
    print()

print("✓ Ensemble voting configured")


In [ ]:
# Cell 5.4: Weight + Threshold Calibration using Training Identity Labels
# ───────────────────────────────────────────────────────────────────────
# For species with training splits (Lynx, Salamander, SeaTurtle) we jointly
# calibrate miew_weight, mega_weight, sift_weight, kaze_weight and
# threshold_cluster by grid-searching AMI against known training identities.
#
# Pipeline:
#   1a. Extract MiewID global embeddings (GPU, TTA).
#   1b. Extract MegaDescriptor global embeddings (GPU, TTA).
#   2. Extract SIFT + KAZE descriptors (CPU, RootSIFT for SIFT).
#   3. Compute BFMatcher match matrices using top-50 global preselection.
#   3b. Extract ALIKED features (GPU) + LightGlue match matrix.
#   4. Grid-search (mw, mgw, sw, kw, thr) with aw = residual to maximise AMI.
#   5. Write optimal weights + threshold back into SPECIES_CONFIG.
#
# THL: no training split → weights + threshold unchanged from defaults.
# Calibrated values are used by Cell 6.2 (AgglomerativeClustering).

import time as _time
from PIL import Image as _PILImg
from sklearn.metrics import adjusted_mutual_info_score as _ami_score

CALIB_SPECIES   = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022"]
CALIB_MAX_IMGS  = {
    "LynxID2025":        9999,  # use all 2957 — more pairs = better weight estimates
    "SalamanderID2025":  9999,  # use all 1388 — 587 IDs, only 36% seen at n=500
    "SeaTurtleID2022":   2000,  # 8729 total, cap at 2000 to keep linkage fast
}
CALIB_BFM_TOPK  = 50    # top-K preselection per image for BFMatcher
CALIB_KPT_CAP   = 300   # max keypoints/image during calibration (speed)
CALIB_CACHE_DIR = Path("/kaggle/working/calib_cache")
CALIB_CACHE_DIR.mkdir(exist_ok=True)

MIEW_W_GRID     = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
MEGA_W_GRID     = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
SIFT_W_GRID     = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
KAZE_W_GRID     = [0, 0.05, 0.10, 0.15, 0.20]   # KAZE explicit (not residual)
# aw = 1 - mw - mgw - sw - kw  (ALIKED is residual; AMI metric)
THR_GRID        = np.linspace(0.15, 0.85, 36).round(4).tolist()  # V5.1: extended to 0.85  # 31 steps

print("=" * 65)
print("V5.1 -- Joint Weight + Threshold Calibration (Fine-tuned MegaDesc + AMI)")
print("=" * 65)
print(f"  Grid: mw={MIEW_W_GRID}  mgw={MEGA_W_GRID}  sw={SIFT_W_GRID}  kw={KAZE_W_GRID}")
print(f"  Thresholds: {THR_GRID[0]:.2f} .. {THR_GRID[-1]:.2f} ({len(THR_GRID)} steps)")
print(f"  Max training images / species: {CALIB_MAX_IMGS}")


# ── Helper 1: global embedding extraction ────────────────────────────────────
def _extract_calib_embs(model, image_paths, image_size, batch_size=64):
    '''Extract L2-normalised MiewID embeddings (original + hflip TTA).'''
    _tfm = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    all_feats = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i : i + batch_size]
            imgs = []
            for p in batch_paths:
                try:
                    img = _PILImg.open(p).convert("RGB")
                    imgs.append(_tfm(img))
                except Exception:
                    imgs.append(torch.zeros(3, image_size, image_size))
            batch = torch.stack(imgs).to(DEVICE)
            with torch.cuda.amp.autocast():
                feats = model(batch) + model(torch.flip(batch, dims=[3]))
            feats = torch.nn.functional.normalize(feats.float(), p=2, dim=1)
            all_feats.append(feats.cpu().numpy())
    return np.concatenate(all_feats).astype(np.float32)


# ── Helper 2: local descriptor extraction (SIFT or KAZE) ─────────────────────
def _extract_calib_local(img_paths, img_keys, ext_type):
    '''Extract SIFT or KAZE descriptors for a list of training image paths.

    img_keys: relative paths (e.g. "LynxID2025/img001.jpg") used for
              get_seg_mask() lookup.  Returns None if not in SAM3 cache.
    RootSIFT is applied to SIFT descriptors (L1-norm + sqrt).
    Returns: list of np.ndarray (N_kpts, D) float32  or  None.
    '''
    if ext_type == "sift":
        det = cv2.SIFT_create(nfeatures=CALIB_KPT_CAP)
    else:
        det = cv2.KAZE_create(upright=False, threshold=0.001,
                              nOctaves=4, nOctaveLayers=4)

    all_descs = []
    for path, key in zip(img_paths, img_keys):
        try:
            img_rgb  = np.array(_PILImg.open(path).convert("RGB"))
            gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
            seg_mask = get_seg_mask(path)  # absolute path → SAM3 + V2.8 yellow filter
            kps, descs = det.detectAndCompute(gray, None)
            if descs is None or len(descs) < 4:
                all_descs.append(None)
                continue
            # Filter keypoints to animal region where mask is available
            if seg_mask is not None:
                h, w = seg_mask.shape[:2]
                pts   = np.array([[int(kp.pt[0]), int(kp.pt[1])] for kp in kps])
                xs    = np.clip(pts[:, 0], 0, w - 1)
                ys    = np.clip(pts[:, 1], 0, h - 1)
                valid = seg_mask[ys, xs] > 0
                descs = descs[valid]
                if len(descs) < 4:
                    all_descs.append(None)
                    continue
            # RootSIFT for SIFT descriptors (KAZE has signed float descs)
            if ext_type == "sift":
                descs = descs.astype(np.float32)
                descs /= (descs.sum(axis=1, keepdims=True) + 1e-7)
                descs  = np.sqrt(descs)
            if len(descs) > CALIB_KPT_CAP:
                descs = descs[:CALIB_KPT_CAP]
            all_descs.append(descs.astype(np.float32))
        except Exception:
            all_descs.append(None)
    return all_descs


# ── Helper 3: BFMatcher match matrix with global top-K preselection ──────────
def _calib_match_matrix(descs_list, global_embs, top_k):
    '''GPU-batched descriptor matching via torch.cdist + Lowe ratio test.

    Replaces CPU BFMatcher loop. All descriptors are stacked on GPU;
    pairs are processed in batches of 200 using torch.cdist. ~100-500x
    faster than sequential OpenCV BFMatcher on CPU.
    Score = 1 - exp(-good_matches / 20).  Ratio threshold = 0.75.
    '''
    N      = len(descs_list)
    matrix = np.zeros((N, N), dtype=np.float32)
    np.fill_diagonal(matrix, 1.0)

    # Collect unique (i<j) pairs
    g_sim = np.clip(global_embs @ global_embs.T, 0.0, 1.0)
    pairs = set()
    for i in range(N):
        sims_i = g_sim[i].copy()
        sims_i[i] = -1.0
        for j in np.argsort(sims_i)[-top_k:]:
            pairs.add((min(i, j), max(i, j)))
    pairs = sorted(pairs)

    # Find valid descriptors
    valid_set = {i for i, d in enumerate(descs_list) if d is not None and len(d) >= 2}
    pairs = [(i, j) for i, j in pairs if i in valid_set and j in valid_set]
    if not pairs:
        return matrix

    dim     = descs_list[next(iter(valid_set))].shape[1]
    max_kp  = max(len(descs_list[i]) for i in valid_set)

    # Stack all descriptors on GPU (N, max_kp, dim)
    _dev = DEVICE
    dt   = torch.zeros(N, max_kp, dim, device=_dev, dtype=torch.float32)
    vm   = torch.zeros(N, max_kp, device=_dev, dtype=torch.bool)
    for i in valid_set:
        d = descs_list[i].astype(np.float32)
        k = min(len(d), max_kp)
        dt[i, :k] = torch.from_numpy(d[:k]).to(_dev)
        vm[i, :k] = True

    # Process pairs in GPU batches (200 pairs per batch)
    _BATCH = 200
    for start in range(0, len(pairs), _BATCH):
        batch = pairs[start : start + _BATCH]
        qi = torch.tensor([p[0] for p in batch], device=_dev)
        ji = torch.tensor([p[1] for p in batch], device=_dev)

        q_d = dt[qi]  # (B, max_kp, dim)
        j_d = dt[ji]  # (B, max_kp, dim)

        # Pairwise L2 distances: (B, max_kp_q, max_kp_db)
        dists = torch.cdist(q_d, j_d)

        # Set padded DB positions to large distance (fail ratio test)
        dists = dists + (~vm[ji]).unsqueeze(1).float() * 1e6

        # Lowe ratio test: 2 nearest DB descriptors per query descriptor
        top2  = dists.topk(2, dim=2, largest=False).values  # (B, max_kp, 2)
        good  = (top2[:, :, 0] < 0.75 * top2[:, :, 1]) & vm[qi]  # (B, max_kp)

        scores = (1.0 - torch.exp(-good.float().sum(dim=1) / 20.0)).cpu().numpy()

        for idx, (i, j) in enumerate(batch):
            s = float(scores[idx])
            matrix[i, j] = s
            matrix[j, i] = s

    return matrix


# ── V3.1 Helper 4: ALIKED feature extraction (GPU) ─────────────────────
def _extract_calib_aliked(img_paths, img_keys, device):
    '''Extract ALIKED features for calibration images.'''
    from lightglue import ALIKED as _ALK
    from lightglue.utils import load_image as _li

    model = _ALK(max_num_keypoints=CALIB_KPT_CAP).eval().to(device)
    all_feats = []
    for path, key in zip(img_paths, img_keys):
        try:
            img = _li(str(path)).to(device)
            with torch.no_grad():
                feats = model.extract(img)
            kps    = feats['keypoints'][0].cpu().numpy()
            descs  = feats['descriptors'][0].cpu().numpy()
            img_sz = feats['image_size'][0].cpu().numpy()

            # Filter by seg_mask (same as SIFT/KAZE calibration)
            seg_mask = get_seg_mask(path)
            if seg_mask is not None:
                h, w = seg_mask.shape[:2]
                xs = np.clip(kps[:, 0].astype(int), 0, w - 1)
                ys = np.clip(kps[:, 1].astype(int), 0, h - 1)
                valid = seg_mask[ys, xs] > 0
                kps   = kps[valid]
                descs = descs[valid]

            if len(kps) < 4:
                all_feats.append(None)
                continue

            all_feats.append({
                'keypoints': kps,
                'descriptors': descs,
                'image_size': img_sz,
            })
        except Exception:
            all_feats.append(None)
    del model
    torch.cuda.empty_cache()
    return all_feats


# ── V3.1 Helper 5: LightGlue match matrix for ALIKED ─────────────────
def _calib_match_matrix_lightglue(feat_list, global_embs, top_k, device):
    '''Build (N, N) match matrix via LightGlue for ALIKED features.'''
    from lightglue import LightGlue as _LG

    N = len(feat_list)
    matrix = np.zeros((N, N), dtype=np.float32)
    np.fill_diagonal(matrix, 1.0)

    matcher = _LG(features='aliked').eval().to(device)

    g_sim = np.clip(global_embs @ global_embs.T, 0.0, 1.0)
    pairs = set()
    for i in range(N):
        sims_i    = g_sim[i].copy()
        sims_i[i] = -1.0
        top_j     = np.argsort(sims_i)[-top_k:]
        for j in top_j:
            pairs.add((min(i, j), max(i, j)))

    for i, j in pairs:
        fi, fj = feat_list[i], feat_list[j]
        if fi is None or fj is None:
            continue
        try:
            d0 = {
                'keypoints':   torch.from_numpy(fi['keypoints']).float().unsqueeze(0).to(device),
                'descriptors': torch.from_numpy(fi['descriptors']).float().unsqueeze(0).to(device),
                'image_size':  torch.tensor(fi['image_size']).float().unsqueeze(0).to(device),
            }
            d1 = {
                'keypoints':   torch.from_numpy(fj['keypoints']).float().unsqueeze(0).to(device),
                'descriptors': torch.from_numpy(fj['descriptors']).float().unsqueeze(0).to(device),
                'image_size':  torch.tensor(fj['image_size']).float().unsqueeze(0).to(device),
            }
            with torch.no_grad():
                result = matcher({'image0': d0, 'image1': d1})
            n_matches    = (result['matches0'][0] > -1).sum().item()
            score        = 1.0 - np.exp(-n_matches / 20.0)
            matrix[i, j] = score
            matrix[j, i] = score
        except Exception:
            pass

    del matcher
    torch.cuda.empty_cache()
    return matrix


# ── Main calibration loop ─────────────────────────────────────────────────────
calib_model = get_global_model()
calib_model.eval()

# V5.1: species-specific fine-tuned model loaded inside loop
calib_mega_model = None  # loaded per-species below

calibrated_config = {}

for sp in CALIB_SPECIES:
    t0 = _time.time()

    sp_train = train_meta[train_meta["dataset"] == sp].copy()
    total_n  = len(sp_train)
    if len(sp_train) > CALIB_MAX_IMGS[sp]:
        sp_train = sp_train.sample(n=CALIB_MAX_IMGS[sp], random_state=42).reset_index(drop=True)

    true_labels, _ = pd.factorize(sp_train["identity"].values)
    n_ids          = len(np.unique(true_labels))
    img_paths      = [os.path.join(ROOT_DIR, p) for p in sp_train["path"].values]
    img_keys       = list(sp_train["path"].values)

    print(f"\n{sp}: {len(sp_train)}/{total_n} train images, {n_ids} identities")

    cfg = SPECIES_CONFIG[sp]

    _cp = str(CALIB_CACHE_DIR / sp)

    # Step 1: global embeddings (cached)
    _gef = Path(f"{_cp}_global_embs.npy")
    if _gef.exists():
        print(f"  [1/6] MiewID embeddings: loaded from cache")
        global_embs = np.load(_gef)
    else:
        print(f"  [1/6] Extracting MiewID embeddings...")
        global_embs = _extract_calib_embs(calib_model, img_paths, cfg["image_size"])
        np.save(_gef, global_embs)
    global_sim = np.clip(global_embs @ global_embs.T, 0.0, 1.0).astype(np.float32)

    # Step 1b: MegaDescriptor global embeddings (cached, V5.1: per-species model)
    _is_ft_calib = _species_has_ft(sp)
    _mega_suffix = "_megaft_global_embs" if _is_ft_calib else "_mega_global_embs"
    _mgef = Path(f"{_cp}{_mega_suffix}.npy")
    _, _mega_img_sz, _ = get_mega_config(sp)
    if _mgef.exists():
        print(f"  [1b/6] MegaDescriptor embeddings: loaded from cache")
        mega_embs = np.load(_mgef)
    else:
        print(f"  [1b/6] Extracting MegaDescriptor embeddings...")
        calib_mega_model = get_mega_model(species=sp)
        mega_embs = _extract_calib_embs(calib_mega_model, img_paths, _mega_img_sz)
        np.save(_mgef, mega_embs)
    mega_sim = np.clip(mega_embs @ mega_embs.T, 0.0, 1.0).astype(np.float32)

    # Step 2: SIFT match matrix (cached)
    _sf = Path(f"{_cp}_sift_matrix.npy")
    if _sf.exists():
        print(f"  [2/6] SIFT match matrix: loaded from cache")
        sift_matrix = np.load(_sf)
    else:
        print(f"  [2/6] Extracting SIFT descriptors + BFMatcher...")
        sift_descs  = _extract_calib_local(img_paths, img_keys, "sift")
        sift_matrix = _calib_match_matrix(sift_descs, global_embs, CALIB_BFM_TOPK)
        np.save(_sf, sift_matrix)

    # Step 3: KAZE match matrix (cached)
    _kf = Path(f"{_cp}_kaze_matrix.npy")
    if _kf.exists():
        print(f"  [3/6] KAZE match matrix: loaded from cache")
        kaze_matrix = np.load(_kf)
    else:
        print(f"  [3/6] Extracting KAZE descriptors + BFMatcher...")
        kaze_descs  = _extract_calib_local(img_paths, img_keys, "kaze")
        kaze_matrix = _calib_match_matrix(kaze_descs, global_embs, CALIB_BFM_TOPK)
        np.save(_kf, kaze_matrix)

    # Step 4: ALIKED match matrix (LightGlue, cached)
    _af = Path(f"{_cp}_aliked_matrix.npy")
    if _af.exists():
        print(f"  [4/6] ALIKED matrix: loaded from cache")
        aliked_matrix = np.load(_af)
    else:
        print(f"  [4/6] Extracting ALIKED features + LightGlue matching...")
        aliked_feats  = _extract_calib_aliked(img_paths, img_keys, DEVICE)
        aliked_matrix = _calib_match_matrix_lightglue(aliked_feats, global_embs, CALIB_BFM_TOPK, DEVICE)
        np.save(_af, aliked_matrix)

    # Step 5: grid search (V4.0: dual-global + KAZE explicit + AMI)
    from scipy.cluster.hierarchy import linkage as _sp_linkage, fcluster as _sp_fcluster
    _n_calib = len(true_labels)
    _triu_idx = np.triu_indices(_n_calib, k=1)

    n_combos = sum(
        1 for mw in MIEW_W_GRID for mgw in MEGA_W_GRID
        for sw in SIFT_W_GRID for kw in KAZE_W_GRID
        if round(1.0 - mw - mgw - sw - kw, 4) >= 0
    )
    print(f"  [5/6] Grid search ({n_combos} weight combos x {len(THR_GRID)} thresholds)...")

    best_mw  = cfg["miew_weight"]
    best_mgw = cfg["mega_weight"]
    best_sw  = cfg["local_weights"]["sift"]
    best_kw  = cfg["local_weights"].get("kaze", 0.0)
    best_aw  = cfg["local_weights"].get("aliked", 0.0)
    best_thr = cfg["threshold_cluster"]
    best_ami = -1.0

    for mw in MIEW_W_GRID:
        for mgw in MEGA_W_GRID:
            for sw in SIFT_W_GRID:
                for kw in KAZE_W_GRID:
                    aw = round(1.0 - mw - mgw - sw - kw, 4)
                    if aw < 0:
                        continue
                    ensemble = (mw * global_sim + mgw * mega_sim
                                + sw * sift_matrix + kw * kaze_matrix
                                + aw * aliked_matrix)
                    dist_sq = np.clip(1.0 - ensemble, 0.0, 1.0).astype(np.float64)
                    Z = _sp_linkage(dist_sq[_triu_idx], method="average")
                    for thr in THR_GRID:
                        pred = _sp_fcluster(Z, t=thr, criterion="distance")
                        ami = _ami_score(true_labels, pred)
                        if ami > best_ami:
                            best_ami = ami
                            best_mw, best_mgw = mw, mgw
                            best_sw, best_kw, best_aw = sw, kw, aw
                            best_thr = thr

    calibrated_config[sp] = {
        "miew_weight":    best_mw,
        "mega_weight":    best_mgw,
        "sift_weight":    best_sw,
        "kaze_weight":    best_kw,
        "aliked_weight":  best_aw,
        "threshold":      best_thr,
        "ami":            best_ami,
    }

    prev_mw  = cfg["miew_weight"]
    prev_mgw = cfg["mega_weight"]
    prev_sw  = cfg["local_weights"]["sift"]
    prev_kw  = cfg["local_weights"].get("kaze", 0.0)
    prev_aw  = cfg["local_weights"].get("aliked", 0.0)
    prev_thr = cfg["threshold_cluster"]
    print(f"  weights:   (mw={prev_mw:.2f}, mgw={prev_mgw:.2f}, sw={prev_sw:.2f}, kw={prev_kw:.2f}, aw={prev_aw:.2f})"
          f" -> (mw={best_mw:.2f}, mgw={best_mgw:.2f}, sw={best_sw:.2f}, kw={best_kw:.2f}, aw={best_aw:.2f})")
    print(f"  threshold: {prev_thr:.2f} -> {best_thr:.2f}"
          f"  (AMI={best_ami:.4f}, t={_time.time()-t0:.1f}s)")

del calib_model, calib_mega_model
_mega_models_cache.clear()
torch.cuda.empty_cache()

# THL: no training split -- keep defaults unchanged
thl_cfg = SPECIES_CONFIG["TexasHornedLizards"]
calibrated_config["TexasHornedLizards"] = {
    "miew_weight":    thl_cfg["miew_weight"],
    "mega_weight":    thl_cfg["mega_weight"],
    "sift_weight":    thl_cfg["local_weights"]["sift"],
    "kaze_weight":    thl_cfg["local_weights"]["kaze"],
    "aliked_weight":  thl_cfg["local_weights"]["aliked"],
    "threshold":      thl_cfg["threshold_cluster"],
    "ami":            None,
}
print(f"\nTexasHornedLizards: no train split -> weights + threshold unchanged")

# Write calibrated weights + thresholds back into SPECIES_CONFIG (used by Cell 6.2)
for sp, cal in calibrated_config.items():
    SPECIES_CONFIG[sp]["miew_weight"]            = cal["miew_weight"]
    SPECIES_CONFIG[sp]["mega_weight"]            = cal["mega_weight"]
    SPECIES_CONFIG[sp]["local_weights"]["sift"]  = cal["sift_weight"]
    if "kaze" in SPECIES_CONFIG[sp]["local_weights"]:
        SPECIES_CONFIG[sp]["local_weights"]["kaze"] = cal["kaze_weight"]
    if "aliked" in SPECIES_CONFIG[sp]["local_weights"]:
        SPECIES_CONFIG[sp]["local_weights"]["aliked"] = cal["aliked_weight"]
    SPECIES_CONFIG[sp]["threshold_cluster"]      = cal["threshold"]

print("\nCalibrated SPECIES_CONFIG:")
for sp, cfg in SPECIES_CONFIG.items():
    lw = cfg["local_weights"]
    print(f"  {sp:<25}: mw={cfg['miew_weight']:.2f}"
          f"  mgw={cfg['mega_weight']:.2f}"
          f"  sw={lw.get('sift', 0):.2f}"
          f"  kw={lw.get('kaze', 0):.2f}"
          f"  aw={lw.get('aliked', 0):.2f}"
          f"  thr={cfg['threshold_cluster']:.2f}")
print("\n✓ Weights + thresholds updated -- Cell 6.2 will use these values")

# Recompute ensemble similarity matrices with calibrated weights.
# Cell 28 built ensemble_similarity_cache using pre-calibration weights;
# this one-liner refreshes it so Cell 6.2 uses the optimal weights.
print("\nRefreshing ensemble_similarity_cache with calibrated weights...")
for _sp in test_meta["dataset"].unique():
    ensemble_similarity_cache[_sp] = compute_ensemble_similarity_matrix(_sp)
    print(f"  ✓ {_sp}")
print("✓ ensemble_similarity_cache refreshed")


## Section 6: Clustering and Submission

In [ ]:
# Cell 6.1: 2-Phase Clustering (Known + Unknown)

def cluster_with_ensemble_scores(species, similarity_matrix, image_ids, threshold):
    """Cluster images using ensemble similarity scores."""
    print(f"\n  Clustering {species}:")
    print(f"    Threshold: {threshold}")
    print(f"    Images: {len(image_ids)}")
    
    # Convert similarity to distance
    dist_matrix = np.clip(1.0 - similarity_matrix, 0, 1)
    
    # Agglomerative clustering
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=threshold,
    )
    labels = clustering.fit_predict(dist_matrix)
    
    n_clusters = len(np.unique(labels))
    print(f"    ✅ Found {n_clusters} individuals")
    
    # Format cluster labels
    cluster_labels = [f"cluster_{species}_{lbl}" for lbl in labels]
    
    return pd.DataFrame({
        "image_id": image_ids,
        "cluster": cluster_labels
    })

print("✓ Clustering function defined")

In [ ]:
# Cell 6.2: Generate Clusters for All Species

results = []

print("="*60)
print("Clustering with Ensemble Scores")
print("="*60)

for species in test_meta["dataset"].unique():
    cfg = SPECIES_CONFIG[species]
    sp_meta = test_meta[test_meta["dataset"] == species]
    
    # Get ensemble similarity matrix
    similarity_matrix = ensemble_similarity_cache[species]
    
    # Cluster
    cluster_df = cluster_with_ensemble_scores(
        species,
        similarity_matrix,
        sp_meta["image_id"].values,
        cfg["threshold_cluster"]
    )
    
    results.append(cluster_df)

print("\n✓ All species clustered")

In [ ]:
# Cell 6.3: Generate Submission CSV

# Combine all results
predictions = pd.concat(results, ignore_index=True)

# Load sample submission
sample_sub = pd.read_csv(os.path.join(ROOT_DIR, "sample_submission.csv"))

# Map predictions to sample submission format
pred_map = dict(zip(predictions["image_id"], predictions["cluster"]))
sample_sub["cluster"] = sample_sub["image_id"].map(pred_map).fillna("cluster_error_0")

# Save submission
sample_sub.to_csv("submission.csv", index=False)

print("\n" + "="*60)
print("🏆 Submission Generated!")
print("="*60)
print(f"Total predictions: {len(sample_sub):,}")
print(f"Total clusters: {sample_sub['cluster'].nunique():,}")
print(f"\nBreakdown by species:")
for species in test_meta["dataset"].unique():
    sp_clusters = predictions[predictions["cluster"].str.contains(species)]["cluster"].nunique()
    print(f"  {species}: {sp_clusters} clusters")

print("\n✓ submission.csv saved")
sample_sub.head(10)